In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE STORAGE INTEGRATION retail_s3_integration
    TYPE = EXTERNAL_STAGE
    STORAGE_PROVIDER = 'S3'
    ENABLED = TRUE
    STORAGE_AWS_ROLE_ARN = '<IAM/Roles/ARN_ID>'
    STORAGE_ALLOWED_LOCATIONS = ('s3://<Your_S3_Bucket_Name>/');

In [ ]:
%%sql -r dataframe_4
DESC INTEGRATION retail_s3_integration

In [ ]:
%%sql -r dataframe_5
CREATE DATABASE IF NOT EXISTS RETAIL_SALES_DB;
CREATE SCHEMA IF NOT EXISTS RETAIL_SALES_DB.RAW;
USE DATABASE RETAIL_SALES_DB;
USE SCHEMA RAW;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE STAGE retail_s3_stage
    URL = 's3://<Your_S3_Bucket_Name>/'
    STORAGE_INTEGRATION = retail_s3_integration;

In [ ]:
%%sql -r dataframe_7
LIST @retail_s3_stage;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
## StructType will mirror what the csv format is (so it matters doing date first, then product, then region, etc) ##
from snowflake.snowpark.types import StructType, StructField, DateType, StringType, IntegerType, DecimalType, FloatType

schema = StructType([
    StructField("date", DateType()),
    StructField("product", StringType()),
    StructField("region", StringType()),
    StructField("units_sold", IntegerType()),
    StructField("unit_price", DecimalType(10, 2)),
    StructField("total_sales", FloatType())
])

df = session.read.schema(schema).option("field_delimiter", ",").option("skip_header", 1).csv("@retail_s3_stage/sales.csv")
df.show()

In [ ]:
df.write.mode("overwrite").save_as_table("RETAIL_SALES_DB.RAW.SALES")

In [ ]:
%%sql -r dataframe_10
SELECT 
    *
FROM RETAIL_SALES_DB.RAW.SALES
LIMIT 10

In [ ]:
from snowflake.snowpark import functions as F

def get_product_sales_leaderboard(df):
    """
    Aggregates total sales by product, sorted by highest revenue first
    """
    try:
        leaderboard_df = (
            df.group_by("PRODUCT")
              .agg(F.sum("TOTAL_SALES").alias("TOTAL_REVENUE"))
              .sort(F.col("TOTAL_REVENUE").desc())
        )
        return leaderboard_df
    except Exception as e:
        print(f"Error computing product sales leaderboard: {e}")
        return None

In [ ]:
leaderboard_df = get_product_sales_leaderboard(df)
leaderboard_df.show()

In [ ]:
def get_regional_sales_comparison(df):
    """
    Total Revenue for each region
    """
    try:
        regional_df = (
            df.group_by("REGION")
              .agg(F.round(F.sum("TOTAL_SALES"), 2).alias("TOTAL_REVENUE"))
        )
        return regional_df
    except Exception as e:
        print(f"Error computing regional sales comparison: {e}")
        return None

In [ ]:
regional_df = get_regional_sales_comparison(df)
regional_df.show()

In [ ]:
"""
Checking totals for grand, product, and region.  They should be the same if I've done everything correctly
"""
grand_total = df.agg(F.round(F.sum("TOTAL_SALES"), 2).alias("GRAND_TOTAL")).collect()[0]["GRAND_TOTAL"]
product_total = leaderboard_df.agg(F.round(F.sum("TOTAL_REVENUE"), 2).alias("SUM_CHECK")).collect()[0]["SUM_CHECK"]
region_total = regional_df.agg(F.round(F.sum("TOTAL_REVENUE"), 2).alias("SUM_CHECK")).collect()[0]["SUM_CHECK"]

print(grand_total, product_total, region_total)

In [ ]:
def get_monthly_trend_analysis(df):
    """
    Aggregates total sales by month, sorted chronologically.
    """
    try:
        monthly_df = (
            df.with_column("SALE_MONTH", F.date_trunc("month", F.col("DATE")))
            .group_by("SALE_MONTH")
            .agg(F.round(F.sum("TOTAL_SALES"), 2).alias("TOTAL_REVENUE"))
            .sort(F.col("SALE_MONTH"))
        )
        return monthly_df
    except Exception as e:
        print(f"Error computing monthly trend analysis: {e}")
        return None

In [ ]:
monthly_df = get_monthly_trend_analysis(df)
monthly_df.show()

In [ ]:
"""
Units sold vs Revenue
"""
def get_units_sold_vs_revenue(df):
    """
    Compares total units sold against total revenue per product,
    to surface products where volume and revenue don't move together.
    """
    try:
        comparison_df = (
            df.group_by("PRODUCT")
              .agg(
                  F.sum("UNITS_SOLD").alias("TOTAL_UNITS"),
                  F.round(F.sum("TOTAL_SALES"), 2).alias("TOTAL_REVENUE")
              )
        )
        return comparison_df
    except Exception as e:
        print(f"Error computing units vs revenue: {e}")
        return None

In [ ]:
unitsvsrevenue_df = get_units_sold_vs_revenue(df)
unitsvsrevenue_df.show()

In [ ]:
"""
Average Orer Value (AOV)
"""
def average_order_value(df, group_by_column=None):
    """
    Calculates average order value (total revenue / total units sold),
    either for the entire dataset or broken down by a given column.
    """
    try:
        if group_by_column:
            aov_df = (
                df.group_by(group_by_column)
                  .agg(F.round(F.sum("TOTAL_SALES") / F.sum("UNITS_SOLD"), 2).alias("AOV"))
            )
        else:
            aov_df = df.agg(F.round(F.sum("TOTAL_SALES") / F.sum("UNITS_SOLD"), 2).alias("AOV"))
        return aov_df
    except Exception as e:
        print(f"Error calculating AOV: {e}")
        return None

In [ ]:
averageordervalue_df = average_order_value(df)
averageordervalue_df.show()

aov_by_region_df = average_order_value(df, "REGION")
aov_by_region_df.show()

aov_by_product_df = average_order_value(df, "PRODUCT")
aov_by_product_df.show()